# Data Flywheel — Evaluation Analysis

Analyse LLM judge scores, latency/cost tradeoffs, and compare ICL vs LoRA SFT across candidate models.

**Prerequisites:** At least one completed flywheel run with evaluated experiments.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGO_DB', 'data_flywheel')

mongo = MongoClient(MONGO_URI)
db = mongo[MONGO_DB]
print('Connected')

## 1. Load Experiments

In [ ]:
experiments = list(db.experiments.find({'status': 'completed'}))
df = pd.DataFrame(experiments)

# Flatten metrics
metrics_df = pd.json_normalize(df['metrics'])
df = pd.concat([df.drop(columns=['metrics']), metrics_df], axis=1)

print(f'Completed experiments: {len(df)}')
print(f'Models: {df["model_id"].unique()}')
print(f'Types: {df["experiment_type"].unique()}')
df[['model_id', 'experiment_type', 'accuracy', 'latency_p95_ms', 'cost_per_1k_tokens_usd']].head(10)

## 2. Judge Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by model
pivot = df.pivot_table(values='accuracy', index='model_id', columns='experiment_type', aggfunc='mean')
pivot.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'], edgecolor='white')
axes[0].set_title('Accuracy (Win Rate) by Model and Experiment Type')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.1)
axes[0].axhline(0.85, color='black', linestyle='--', label='Promotion threshold')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

# Mean score distribution
for exp_type in df['experiment_type'].unique():
    subset = df[df['experiment_type'] == exp_type]['accuracy']
    axes[1].hist(subset, bins=10, alpha=0.6, label=exp_type, edgecolor='white')
axes[1].set_title('Accuracy Distribution by Experiment Type')
axes[1].set_xlabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Latency vs Cost Tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'qwen-0.5b': '#2ecc71', 'qwen-1.5b': '#3498db', 'qwen-3b': '#9b59b6'}
markers = {'icl': 'o', 'lora_sft': 's'}

for _, row in df.iterrows():
    if pd.notna(row.get('latency_p95_ms')) and pd.notna(row.get('cost_per_1k_tokens_usd')):
        color = colors.get(row['model_id'], 'gray')
        marker = markers.get(row['experiment_type'], 'o')
        ax.scatter(row['latency_p95_ms'], row['cost_per_1k_tokens_usd'],
                   c=color, marker=marker, s=150, alpha=0.8, edgecolors='white', linewidth=0.5)

ax.axvline(1500, color='red', linestyle='--', alpha=0.5, label='Max latency (1500ms)')
ax.set_xlabel('Latency p95 (ms)')
ax.set_ylabel('Cost per 1k tokens (USD)')
ax.set_title('Latency vs Cost by Model and Experiment Type')

legend_elements = [mpatches.Patch(color=c, label=m) for m, c in colors.items()]
legend_elements += [plt.Line2D([0], [0], marker=mk, color='gray', label=et, linestyle='None', markersize=8)
                    for et, mk in markers.items()]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

## 4. ICL vs LoRA SFT Comparison

In [ ]:
comparison = df.groupby(['model_id', 'experiment_type']).agg({
    'accuracy': 'mean',
    'latency_p95_ms': 'mean',
    'cost_per_1k_tokens_usd': 'mean',
}).round(4)

print('ICL vs LoRA SFT — Mean Metrics:')
comparison

## 5. Promotion Criteria Check

In [ ]:
criteria = {
    'min_accuracy': 0.85,
    'max_latency_p95_ms': 1500,
    'max_cost_per_1k_tokens_usd': 0.02,
    'min_sample_count': 50,
}

def check_criteria(row):
    return (
        row.get('accuracy', 0) >= criteria['min_accuracy'] and
        row.get('latency_p95_ms', 9999) <= criteria['max_latency_p95_ms'] and
        row.get('cost_per_1k_tokens_usd', 9999) <= criteria['max_cost_per_1k_tokens_usd']
    )

df['passes_criteria'] = df.apply(check_criteria, axis=1)

print(f"Experiments passing promotion criteria: {df['passes_criteria'].sum()} / {len(df)}")
df[['model_id', 'experiment_type', 'accuracy', 'latency_p95_ms', 'cost_per_1k_tokens_usd', 'passes_criteria']]